# 01 — Unitaries, Spectrum, and Dynamics

**Trilha:** Fundamentos operacionais + Pensamento espectral  
**Milestone:** 1 — Linguagem mínima do problema quântico

---

## Pergunta

Por que a evolução precisa ser unitária? O que o espectro de um operador unitário diz sobre a dinâmica que ele gera?

Unitaridade não é uma restrição arbitrária de hardware. É uma consequência estrutural de exigir que normas sejam preservadas e que evolução seja reversível. Entender por que isso importa é entender o que distingue computação quântica de computação estocástica.

## Modelo matemático mínimo

### Operador unitário
Um operador $U$ é unitário se $U^\dagger U = U U^\dagger = I$.

Consequências imediatas:
- **Preserva norma:** $\|U|\psi\rangle\| = \||\psi\rangle\|$
- **É reversível:** $U^{-1} = U^\dagger$
- **Autovalores estão na circunferência unitária:** $|\lambda| = 1$, logo $\lambda = e^{i\theta}$

### Relação Hamiltoniano-Unitário
Todo unitário pode ser escrito como $U = e^{iH}$ para algum Hermitiano $H$. Os autovalores de $U$ são $e^{i\lambda_k}$ onde $\lambda_k$ são os autovalores de $H$.

Isso é central: o espectro de $U$ (as fases $e^{i\theta_k}$) codifica o espectro de energia de $H$. **Phase estimation lê exatamente esse código.**

### Dinâmica
A evolução temporal por tempo $t$ de um estado $|\psi\rangle = \sum_k c_k |u_k\rangle$ (expresso na base de autovetores de $H$):
$$U(t)|\psi\rangle = e^{-iHt}|\psi\rangle = \sum_k c_k e^{-i\lambda_k t} |u_k\rangle$$

Cada componente gira com frequência proporcional ao seu autovalor.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

# --- O que quebra quando a evolução NÃO é unitária ---

# Operador unitário: rotação
theta = np.pi / 4
U = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])

# Operador não-unitário: escala assimétrica
A = np.array([[1.2, 0.0],
              [0.0, 0.8]])

state = np.array([1.0, 0.0])  # estado inicial

# Aplicar repetidamente
n_steps = 20
norms_U = [np.linalg.norm(np.linalg.matrix_power(U, k) @ state) for k in range(n_steps)]
norms_A = [np.linalg.norm(np.linalg.matrix_power(A, k) @ state) for k in range(n_steps)]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(norms_U, label='Unitário (rotação)', marker='o', markersize=4)
ax.plot(norms_A, label='Não-unitário (escala)', marker='s', markersize=4)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5, label='norma = 1')
ax.set_xlabel('Passos de evolução')
ax.set_ylabel('Norma do estado')
ax.set_title('Preservação de norma: unitário vs. não-unitário')
ax.legend()
plt.tight_layout()
plt.show()

print('Unitário: norma preservada (probabilidades somam 1 sempre)')
print('Não-unitário: norma deriva (interpretação probabilística quebra)')

In [ ]:
# --- Espectro de um unitário: autovalores na circunferência ---
# Construir unitário a partir de Hamiltoniano
H = np.array([[0, 1], [1, 0]], dtype=complex)  # Pauli X como Hamiltoniano
t = np.pi / 4
U_from_H = expm(-1j * H * t)

eigvals = np.linalg.eigvals(U_from_H)
H_eigvals = np.linalg.eigvalsh(H)

print('Hamiltoniano H = X (Pauli X)')
print(f'Autovalores de H: {H_eigvals}')  # -1, +1
print(f'Autovalores de U = exp(-iHt) com t = π/4: {eigvals}')
print(f'|autovalores de U|: {np.abs(eigvals)}')  # deve ser 1
print()

# Fases dos autovalores de U
phases = np.angle(eigvals) / (2 * np.pi)  # em unidades de 2π
expected_phases = -H_eigvals * t / (2 * np.pi)
print(f'Fases de U (em unidades de 2π): {phases}')
print(f'Fases esperadas (-λ_H * t / 2π):  {expected_phases}')
print()
print('Conclusão: as fases dos autovalores de U CODIFICAM os autovalores de H.')
print('Phase estimation lê exatamente essas fases.')

In [ ]:
# --- Dinâmica: evolução de um estado na base de autovetores ---
# Estado inicial: superposição dos autovetores de H
eigvals_H, eigvecs_H = np.linalg.eigh(H)
initial_state = (eigvecs_H[:, 0] + eigvecs_H[:, 1]) / np.sqrt(2)

# Evolução temporal
times = np.linspace(0, 4 * np.pi, 300)
probs_0 = []

for t_val in times:
    U_t = expm(-1j * H * t_val)
    evolved = U_t @ initial_state
    prob = np.abs(evolved[0])**2  # P(|0>)
    probs_0.append(prob)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(times / np.pi, probs_0, color='steelblue')
ax.set_xlabel('Tempo (unidades de π)')
ax.set_ylabel('P(|0⟩)')
ax.set_title('Dinâmica: oscilação determinada pelo espectro de H')
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

# Frequência da oscilação
freq = (eigvals_H[1] - eigvals_H[0]) / (2 * np.pi)
print(f'Separação espectral de H: Δλ = {eigvals_H[1] - eigvals_H[0]:.3f}')
print(f'Frequência de oscilação esperada: {freq:.3f} ciclos por unidade de tempo')
print('A dinâmica é completamente determinada pelo espectro de H.')

## Conclusão

1. **Unitaridade não é opcional.** Sem ela, a norma do estado deriva e a interpretação probabilística da medição quebra. Um operador que não preserva norma não é uma operação quântica válida.

2. **Autovalores de um unitário estão na circunferência.** $\lambda = e^{i\theta}$ — carregam informação de fase, não de módulo.

3. **O espectro de $H$ determina completamente a dinâmica de $e^{-iHt}$.** A oscilação das probabilidades tem frequência proporcional à diferença de autovalores. Isso é um fato espectral, não um acidente de simulação.

4. **Phase estimation é a extração dessas fases por circuito.** O próximo notebook mostra o mecanismo que torna isso possível: phase kickback.

---

**Pergunta aberta para o próximo notebook:** como um circuito consegue transferir informação de fase de um operador para um registrador que pode ser medido?